in this notebook, we load all the dataset files, clean the columns and dates, compute the daily returns for each asset, merge everything by date, fill missing values, and run our multiple ols regression to explain spy returns.

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
from statsmodels.stats.outliers_influence import variance_inflation_factor

# import the csv into dataframe
data1 = pd.read_csv("../../data/Données SPY.csv")
data2 = pd.read_csv("../../data/AAPL.csv")
data3 = pd.read_csv("../../data/USD_CNY.csv")
data4 = pd.read_csv("../../data/DTB6.csv")
data5 = pd.read_csv("../../data/HIS.csv")
data6 = pd.read_csv("../../data/Gold.csv")
data7 = pd.read_csv("../../data/TCD6.csv")
data8 = pd.read_csv("../../data/DR5.xls")
data9 = pd.read_csv("../../data/Taux_ST6.xls")
data10 = pd.read_csv("../../data/VRV.xls")

# standardize columns and convert date
for df in [data1, data2, data3, data4, data5, data6, data7, data8, data9, data10]:
    df.columns = [col.strip().capitalize() for col in df.columns]
    if 'Unnamed: 0' in df.columns:
        df.rename(columns={'Unnamed: 0':'Date'}, inplace=True)
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

# compute returns before merging
# specific cleaning for the his file
data5['Price'] = data5['Price'].astype(str)
data5['Price'] = data5['Price'].str.replace(',', '', regex=False)  # remove commas
data5['Price'] = pd.to_numeric(data5['Price'], errors='coerce')

col_names = ['SPY','AAPL','DollarYuan','DTB6','HIS','Or','DCD6M', 'DR5', 'Taux_ST6', 'VRV']
col_prices = ['Price','Price','Price','Dtb6','Price','Price','Dcd6m','Price', 'Price', 'Price']
dataframes = [data1, data2, data3, data4, data5, data6, data7, data8, data9, data10]

for df, name, price in zip(dataframes, col_names, col_prices):
    if price in df.columns:
        # convert column to numeric before calculation
        df[price] = pd.to_numeric(df[price], errors='coerce')
        # compute daily return in proportion
        df[name] = df[price].pct_change()
        # drop first nan rows
        df.dropna(subset=[name], inplace=True)

# merge files on date column with outer join
# start from spy already converted to return
data_merged = data1[['Date','SPY']]

# list of remaining files to merge
dfs = [data2, data3, data4, data5, data6, data7, data8, data9, data10]
col_names = ['AAPL','DollarYuan','DTB6','HIS','Or','DCD6M','DR5', 'Taux_ST6', 'VRV' ]
col_prices = ['AAPL','DollarYuan','DTB6','Price','Or','DCD6M','Dr5', 'St6', 'Vrv']  # merge on already computed returns

for df, col_name, price_col in zip(dfs, col_names, col_prices):
    data_merged = data_merged.merge(df[['Date', price_col]], on='Date', how='outer')
    data_merged.rename(columns={price_col: col_name}, inplace=True)

print("number of line after merge :", len(data_merged))

# fill missing values
data_merged.ffill(inplace=True)
data_merged.bfill(inplace=True)
for col in ['SPY','AAPL','DollarYuan','DTB6','HIS','Or','DCD6M','DR5', 'Taux_ST6', 'VRV' ]:
    data_merged[col] = pd.to_numeric(data_merged[col], errors='coerce')
    if data_merged[col].isna().sum() > 0:
        data_merged[col] = data_merged[col].fillna(data_merged[col].mean())

# drop columns with more than 50% nan
X_multi = data_merged[['AAPL','DollarYuan','DTB6','HIS','Or','DCD6M','DR5', 'Taux_ST6', 'VRV']]
X_multi = X_multi.loc[:, X_multi.isna().mean() < 0.5]
y = data_merged[['SPY']]

# final cleaning: drop remaining rows with nan
data_clean = pd.concat([X_multi, y], axis=1).dropna()
X_multi = data_clean.drop(columns=['SPY'])
y = data_clean[['SPY']]

print("final lines after cleaning :", len(data_clean))
print("final columns :", X_multi.columns.tolist())

# multiple regression
X_multi_sm = sm.add_constant(X_multi)
mon_modele = sm.OLS(y, X_multi_sm).fit()
print(mon_modele.summary())

number of line after merge : 1302
final lines after cleaning : 1302
final columns : ['AAPL', 'DollarYuan', 'DTB6', 'HIS', 'Or', 'DCD6M', 'DR5', 'Taux_ST6', 'VRV']
                            OLS Regression Results                            
Dep. Variable:                    SPY   R-squared:                       0.149
Model:                            OLS   Adj. R-squared:                  0.143
Method:                 Least Squares   F-statistic:                     25.06
Date:                Tue, 24 Nov 2026   Prob (F-statistic):           5.54e-40
Time:                        00:20:08   Log-Likelihood:                 3860.5
No. Observations:                1302   AIC:                            -7701.
Df Residuals:                    1292   BIC:                            -7649.
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t    

looking at the regression results, we have 1305 observations and an r-squared of 0.15 (so the model explains around 15% of spy daily returns). we can see that aapl is the only really significant variable here (p-value < 0.001) with a positive effect on spy, while all the other variables have high p-values (> 0.05) and are not statistically significant. also, the jarque-bera test p-value is 0, which shows that the residuals are not normally distributed.